# The MICADO Calibration Assembly (MCA)

This notebook describes how to simulate calibration exposures using the MICADO Calibration Assembly with ScopeSim. For this purpose a new submode `CALIB` has been introduced, which replaces the `SCAO` and `MCAO` submodes that are used for on-sky observations. It can be combined with all instrument modes, i.e. `IMG_4mas`, `SPEC`, etc. 
`CALIB` does not include the Armazones (atmosphere) and ELT effects. It currently includes the following to describe the MCA:
- `mca_mirror`: a single deployable mirror that is unique to the MCA. As with all mirrors the effect describes throughput (reflectivity) as well as thermal emission.
- `relay_surface_list`: This is the list of mirrors in the relay optics that is used in stand-alone mode, identical to the mirror list in the `SCAO` submode. Note that MORFEO is not yet supported for MCA simulations.
- `air_transmission`: The optical path from the MCA to the entrance window of MICADO has a length of about 14 metres through air, which therefore imprints an absorption signal on the input (continuum) spectrum. The effect uses a library of transmission spectra for various values of relative humidity (see below for details).
- `psf`: Very simplistically, the instrumental PSF (imprinted on observations using a pinhole mask) is modeled as Gaussian PSF of FWHM = 0.02 arcsec. This can be made more realistic in the future.

In [ ]:
import scopesim as sim

In [ ]:
sim.link_irdb("../../../")

If you have not done so already, please download the relevant instrument packages using the following code in a new cell:

```sim.download_packages(["MICADO"])```

Alternatively, if you would like to keep the instrument packages in a separate directory, you can set the following config value:

```sim.set_inst_pkgs_path("path/to/packages")```

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
from astropy.wcs import WCS

## Setting up the optical train
We first set up MICADO for the nominal imaging mode:

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "IMG_4mas"])

In [ ]:
micado = sim.OpticalTrain(cmd)

In [ ]:
micado.effects.pprint_all()

## Imaging 
The MICADO calibration mode needs to use a `Source` object (unlike the METIS WCU mode). For the time being, we use a flat field from `Scopesim_Templates`. This will soon change to `Scopesim_Targets`, which will also allow implementation of pinhole masks.

In [ ]:
from scopesim_templates.micado import flatlamp

In [ ]:
flat = flatlamp()

In [ ]:
micado.observe(flat)

In [ ]:
readout = micado.readout(dit=1, ndit=1)[0]

In [ ]:
print("Mean:     ", readout[1].data.mean())
print("Std. dev.:", readout[1].data.std())

In [ ]:
plt.hist(readout[1].data.ravel(), bins=100);

Also from scopesim_templates, a pinhole mask, which we shall set up as a regular grid for imaging (blindly taken from the documentation).

In [ ]:
from scopesim_templates.micado.pinhole_masks import pinhole_mask

In [ ]:
dr = np.arange(-5, 6, 0.5)      # [arcsec]
x, y = np.meshgrid(dr, dr)
x, y = x.flatten(), y.flatten()
waves = np.arange(0.7, 2.5, 0.001) * u.um
pinh = pinhole_mask(x, y, waves, sum_factor=9001)

In [ ]:
micado.observe(pinh)

In [ ]:
read_pinh = micado.readout(dit=1, ndit=1)[0]

In [ ]:
plt.imshow(read_pinh[1].data)
plt.title("Imaging, 4 mas");

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "IMG_1.5mas"])

In [ ]:
micado = sim.OpticalTrain(cmd)
micado.observe(pinh)
read_pinh_zoom = micado.readout(dit=1, ndit=1)[0]

In [ ]:
plt.imshow(read_pinh_zoom[1].data)
plt.title("Imaging, 1.5 mas");

# Spectroscopy

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "SPEC"])
micado = sim.OpticalTrain(cmd)

In [ ]:
micado.effects.pprint_all()

We will observe the flat lamp, which allows us to switch off the psf effects. We'll try to simulate a full field of view.

In [ ]:
micado['psf'].include = False
micado['micado_ncpas_psf'].include = False
micado['filter_wheel_1'].change_filter("Spec_HK")
micado['detector_window'].include = False
micado['full_detector_array'].include = True

The transmission of the 14 meter air column in the MCA and relay optics is provided by the `air_transmission` effect. This is a library of transmission spectra for relative humidities between 5 and 95 per cent, available in steps of 5 per cent. The default is 10 per cent, which can be changed with the `update()` method:

In [ ]:
air = micado['air_transmission']
print("Default humidity:", air.meta['relH'], "(per cent)")

In [ ]:
air.update(relH=50)
print("Current humidity:", air.meta['relH'], "(per cent)")

In [ ]:
air.plot();

In [ ]:
micado.observe(flat)

In [ ]:
readout = micado.readout(dit=600, ndit=1)[0]

In [ ]:
plt.imshow(readout[5].data)

In [ ]:
rect = micado['micado_spectral_traces'].rectify_traces(readout, -1.5, 1.5)

In [ ]:
plt.imshow(rect[1].data); 
plt.xlim(500, 4000);

In [ ]:
j = np.arange(rect[2].data.shape[1])
wcs = WCS(rect[2].header).spectral
lam = wcs.all_pix2world(j, 0)[0]
lam = (lam * wcs.wcs.cunit[0]).to(u.um)
plt.plot(lam, rect[2].data[300, ])
plt.title(rect[2].header["EXTNAME"])
plt.xlabel("Wavelength [um]");